In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/YOUTUBE/Final_Mode.csv')

In [ ]:
data = data.dropna()

In [ ]:
data

,Unnamed: 0,FaceRectX,FaceRectY,FaceRectWidth,FaceRectHeight,FaceScore,x_0,x_1,x_2,x_3,...,disgust,fear,happiness,sadness,surprise,neutral,input,frame,question,answer
0,0,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Are there signs of Unusual/ excessive smiling?,no
1,1,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Does the child have reduced eye contact?,no
2,2,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Are their lips parted?,no
3,3,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Is the face asymmetric(FA)?,no
4,4,291.363858,35.447479,93.662453,119.462808,0.999587,290.202567,289.394182,289.544206,291.524437,...,0.002833,0.001497,0.857664,0.006462,0.019306,0.109500,/content/data/frame0.jpg,0,Are cheeks puffed or raised with happiness?,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346088,346088,1049.286567,355.978798,212.472548,325.662575,0.962633,1026.879664,1044.788952,1064.371926,1086.603797,...,0.001116,0.041112,0.034277,0.451203,0.060943,0.322786,/content/data/frame859.jpg,0,Does the lip corner depressor show discomfort ...,no
346089,346089,1049.286567,355.978798,212.472548,325.662575,0.962633,1026.879664,1044.788952,1064.371926,1086.603797,...,0.001116,0.041112,0.034277,0.451203,0.060943,0.322786,/content/data/frame859.jpg,0,Are there signs of jaw clenching or teeth grin...,no
346090,346090,1049.286567,355.978798,212.472548,325.662575,0.962633,1026.879664,1044.788952,1064.371926,1086.603797,...,0.001116,0.041112,0.034277,0.451203,0.060943,0.322786,/content/data/frame859.jpg,0,Does the child have atypical gaze patterns?,yes
346091,346091,1049.286567,355.978798,212.472548,325.662575,0.962633,1026.879664,1044.788952,1064.371926,1086.603797,...,0.001116,0.041112,0.034277,0.451203,0.060943,0.322786,/content/data/frame859.jpg,0,Are the inner portions of the brows furrowed h...,yes


## LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Concatenate

# Prepare the data
questions = data['question'].values
print(questions)
answers = data['answer'].values
features = data[[
    'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11', 'AU12',
    'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26', 'AU28', 'AU43',
    'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral'
]].values

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(questions)
question_sequences = tokenizer.texts_to_sequences(questions)

# Pad sequences to have the same length
max_sequence_length = max(len(seq) for seq in question_sequences)
question_sequences = pad_sequences(question_sequences, maxlen=max_sequence_length, padding='post')

# Prepare the target labels
labels = np.array([1 if ans.lower() == 'yes' else 0 for ans in answers])

# Define the model architecture
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

input_text = Input(shape=(max_sequence_length,))
embedding = Embedding(vocab_size, embedding_dim)(input_text)
lstm = LSTM(100)(embedding)

input_features = Input(shape=(27,))
concat = Concatenate()([lstm, input_features])
output = Dense(1, activation='sigmoid')(concat)

model = Model(inputs=[input_text, input_features], outputs=output)

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit([question_sequences, features], labels, epochs=10, batch_size=32)

# Save the model
model.save('question_answering_model.h5')


['Are there signs of Unusual/ excessive smiling?'
 ' Does the child have reduced eye contact?' 'Are their lips parted?' ...
 'Does the child have atypical gaze patterns?'
 'Are the inner portions of the brows furrowed horizontally?'
 'Is there absence of the inner brow raiser?']
Epoch 1/10
10753/10753 [==============================] - 284s 26ms/step - loss: 0.5137 - accuracy: 0.7664
Epoch 2/10
10753/10753 [==============================] - 280s 26ms/step - loss: 0.5006 - accuracy: 0.7710
Epoch 3/10
10753/10753 [==============================] - 276s 26ms/step - loss: 0.4995 - accuracy: 0.7709
Epoch 4/10
10753/10753 [==============================] - 282s 26ms/step - loss: 0.4991 - accuracy: 0.7696
Epoch 5/10
10753/10753 [==============================] - 281s 26ms/step - loss: 0.4989 - accuracy: 0.7696
Epoch 6/10
10753/10753 [==============================] - 280s 26ms/step - loss: 0.4989 - accuracy: 0.7700
Epoch 7/10
10753/10753 [==============================] - 278s 26ms/step - los

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## Testing

In [ ]:
import json
tokenizer = tokenizer

tokenizer_json = tokenizer.to_json()

with open('tokenizer.json', 'w') as json_file:
    json_file.write(tokenizer_json)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

# Load the saved model
model = load_model('question_answering_model.h5')

# Load the tokenizer
tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(open('tokenizer.json').read())

# Load the maximum sequence length
max_sequence_length = model.input_shape[0][1]

# Function to preprocess input question
def preprocess_question(question):
    question_sequence = tokenizer.texts_to_sequences([question])
    question_sequence = pad_sequences(question_sequence, maxlen=max_sequence_length, padding='post')
    return question_sequence

# Function to preprocess input features
def preprocess_features(features):
    return np.array([features])

# Function to decode predicted answer

def decode_answer(answer):
    if answer > 0.5:
        return 'yes'
    else:
        return 'no'

# Example input features and question
input_features = [0.28658053, 0.0, 0.672386, 0, 0.09298704, 0, 0.12514895, 0.031605154, 0.4653802, 0.08947913, 0.2521626, 0.16241404, 0.4412326, 0, 0.41045642, 0.40597105, 0.95056677, 0.38891232, 0.007265171, 0.039709873, 0.00540928, 0.000198417, 0.001921368, 0.000234943, 0.002749274, 0.7617219, 0.22776474]



input_question = "Is the inner brow raiser not there?"
# input_question = "Does the child have atypical gaze patterns?"

# Preprocess the input features and question
processed_features = preprocess_features(input_features)
processed_question = preprocess_question(input_question)

# Make predictions
predicted_answer = model.predict([processed_question, processed_features])
decoded_answer = decode_answer(predicted_answer[0][0])
print(predicted_answer)

# Print the predicted answer
print("Predicted Answer:", decoded_answer)


1/1 [==============================] - 0s 84ms/step
[[0.47594672]]
Predicted Answer: no
